In [ ]:
%pip install -U langchain-classic langchain-community rank_bm25

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain.retrievers (from versions: none)
ERROR: No matching distribution found for langchain.retrievers


In [15]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings

# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import sys

print(sys.executable)
print(sys.version)

d:\proj\ecom-ai-project\ecom-ai-backend\venv\Scripts\python.exe
3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [9]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [14]:
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

for i, chunk in enumerate(chunks,1):
    print(f"{i}. {chunk}")

print("\n" + "="*80)



1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java programming req

In [39]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},

    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size":2
    }
)

vectorestore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
vector_retriever = vectorestore.as_retriever(search_kwargs={"k": 2})

test_query = "space exploration company"

test_docs = vector_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")


Found: SpaceX develops Starship rockets for Mars missions.
Found: SpaceX develops Starship rockets for Mars missions.


In [ ]:

# 2. BM25 Retriever (Keyword Search)
bm25_retriever = BM25Retriever.from_documents(documents)

bm25_retriever.k = 3

In [ ]:
# Test exact keyword matching
# test_query = "space exploration company"

test_query = "Cybertruck"

test_docs = bm25_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")

Found: Cybertruck reservations exceeded company expectations.
Found: Tesla Cybertruck production ramp begins in 2024.
Found: Orange juice sales increased during winter months.


In [37]:
#  3. Hybrid Retriever (Combination)
print("Setting up Hybrid Retriever...")

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

Setting up Hybrid Retriever...


In [40]:
# Query 1: Mixed semantic and exact terms

# Vector search understands "purchase cost" semantically
# BM25 search finds exact "7.5 billion" 
# Hybrid should combine both strengths for best result
test_query = "purchase cost 7.5 billion"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching")

1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Orange County reported new housing developments.
3. Orange juice sales increased during winter months.

Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching


In [41]:
# Query 2: Semantic concept + specific product name  

# Vector search understands "electric vehicle manufacturing"
# BM25 search finds exact "Cybertruck"
# Hybrid gets the best of both worlds

test_query = "electric vehicle manufacturing Cybertruck"

retrieved_chunks = hybrid_retriever.invoke(test_query)

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 2 demonstrates combining product-specific terms with broader concepts")

1. Tesla Cybertruck production ramp begins in 2024.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.

Query 2 demonstrates combining product-specific terms with broader concepts


In [42]:
# Query 3: Where neither alone would be perfect

# "Company performance" is semantic, "Tesla" is exact keyword
# Hybrid should find the most relevant Tesla performance info

test_query = "company performance Tesla"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 3 shows how hybrid handles mixed semantic/keyword queries better than either approach alone")

1. Tesla Tesla Tesla financial quarterly results improved significantly.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.

Query 3 shows how hybrid handles mixed semantic/keyword queries better than either approach alone


In [44]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

# Create a ChatOpenAI model
# model = ChatOpenAI(model="gpt-4o")
model = ChatOllama(
    model="qwen3:0.6b",
    base_url="http://localhost:11434",
    temperature=0
)

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

# Display the full result and content only
print("\n--- Generated Response ---")
# print("Full result:")
# print(result)
print("Content only:")
print(result.content)


--- Generated Response ---
Content only:
Based on the provided documents, Tesla's company performance is highlighted as follows:  
- Financial quarterly results improved significantly.  
- Tesla reported strong quarterly results, continuing to lead in electric vehicles.  
- Tesla announced new manufacturing facilities.  
- Cybertruck reservations exceeded company expectations.  

All relevant information is included.
